# 🎓 Face Recognition Attendance System
### Built with DeepFace + FaceNet512 | Google Colab Ready

**Workflow:**
1. Install dependencies
2. Download & use pre-trained FaceNet512 model (trained on VGGFace2 — millions of faces)
3. Upload your `ABCD.zip` (176 student photos, 1 per person)
4. Build face-embedding database for each student
5. Test recognition on new images / webcam snapshots
6. Auto-mark attendance in a CSV with timestamp

> ℹ️ This uses **embedding-based recognition** (not retraining). The model learns to compare
> faces on a massive public dataset; your student photos are used only as reference templates.


## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install deepface tf-keras opencv-python-headless tqdm -q
!pip install gdown -q
print("✅ All packages installed")


## 🔧 Step 2 — Imports & Config

In [ ]:
import os, re, zipfile, shutil, pickle, warnings
import numpy as np
import pandas as pd
import cv2
from datetime import datetime
from tqdm import tqdm
from deepface import DeepFace
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, Image as IPImage, HTML
from google.colab import files
warnings.filterwarnings("ignore")

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME     = "Facenet512"      # Best accuracy for 1-shot recognition
DETECTOR       = "retinaface"      # Most accurate face detector
DISTANCE_METRIC= "cosine"          # Cosine similarity for embeddings
THRESHOLD      = 0.40              # Lower = stricter match (0.0–1.0)
DB_PATH        = "/content/face_db"
ATTENDANCE_CSV = "/content/attendance.csv"
EMBED_CACHE    = "/content/embeddings.pkl"

os.makedirs(DB_PATH, exist_ok=True)
print("✅ Configuration ready")
print(f"   Model    : {MODEL_NAME}")
print(f"   Detector : {DETECTOR}")
print(f"   Threshold: {THRESHOLD}")


## 📁 Step 3 — Upload ABCD.zip (Your Student Dataset)

In [ ]:
print("📤 Please upload your ABCD.zip file...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {zip_name}")

# Extract
EXTRACT_DIR = "/content/student_photos"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(EXTRACT_DIR)
print(f"✅ Extracted to {EXTRACT_DIR}")

# List all image files
img_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
all_files = []
for root, dirs, files_list in os.walk(EXTRACT_DIR):
    for f in files_list:
        if os.path.splitext(f)[1].lower() in img_extensions:
            all_files.append(os.path.join(root, f))

print(f"📸 Found {len(all_files)} image files")


## 🏷️ Step 4 — Extract Student Names from Filenames

In [ ]:
def extract_name(filepath):
    """
    Filename convention:  <anything> - STUDENT NAME.ext
    OR just:             STUDENT NAME.ext
    Returns TITLE CASED name.
    """
    filename = os.path.basename(filepath)
    name_part = os.path.splitext(filename)[0]

    # Try ' - NAME' pattern (most common)
    if ' - ' in name_part:
        name = name_part.split(' - ')[-1].strip()
    else:
        name = name_part.strip()

    # Clean up: remove leading 'MS.' or similar prefixes
    name = re.sub(r'^(MS|MR|DR)\.?\s*', '', name, flags=re.IGNORECASE).strip()
    return name.title()

# Build student registry
student_registry = {}
skipped = []

for fpath in all_files:
    ext = os.path.splitext(fpath)[1].lower()
    if ext == '.pdf':          # skip PDF entries
        skipped.append(fpath)
        continue
    name = extract_name(fpath)
    if name and len(name) > 1:
        if name not in student_registry:
            student_registry[name] = fpath
        # If duplicate name, keep first occurrence
print(f"✅ Parsed {len(student_registry)} unique students")
if skipped:
    print(f"⚠️  Skipped {len(skipped)} non-image file(s): {[os.path.basename(s) for s in skipped]}")

# Preview
preview_df = pd.DataFrame([
    {"Student Name": k, "Photo File": os.path.basename(v)}
    for k, v in list(student_registry.items())[:10]
])
print("\n📋 Sample (first 10):")
display(preview_df)
print(f"\n... and {len(student_registry)-10} more students")


## 🖼️ Step 5 — Preview Sample Student Photos

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
fig.suptitle("Sample Student Photos from ABCD.zip", fontsize=16, fontweight='bold')
axes = axes.flatten()

sample_students = list(student_registry.items())[:15]
for i, (name, path) in enumerate(sample_students):
    img = cv2.imread(path)
    if img is None:
        axes[i].text(0.5, 0.5, 'Cannot\nload', ha='center', va='center')
        axes[i].set_title(name, fontsize=9)
        axes[i].axis('off')
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Resize for display
    h, w = img_rgb.shape[:2]
    max_dim = 200
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img_rgb = cv2.resize(img_rgb, (int(w*scale), int(h*scale)))
    axes[i].imshow(img_rgb)
    axes[i].set_title(name, fontsize=9, fontweight='bold')
    axes[i].axis('off')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig("/content/sample_photos.png", dpi=100, bbox_inches='tight')
plt.show()
print("✅ Sample photos displayed")


## 🧠 Step 6 — Build Face Embedding Database
This is the "registration" phase. Each student photo is passed through FaceNet512
to produce a 512-dimensional embedding vector that uniquely represents their face.
This only needs to run once — embeddings are cached to disk.


In [ ]:
def build_embedding_database(registry, model_name, detector, cache_path):
    """Generate and cache face embeddings for all students."""

    if os.path.exists(cache_path):
        print(f"✅ Loading cached embeddings from {cache_path}")
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    print(f"🔄 Building embeddings for {len(registry)} students...")
    print("   (First run downloads FaceNet512 weights ~90 MB — please wait)\n")

    embeddings = {}
    failed = []

    for name, img_path in tqdm(registry.items(), desc="Processing students"):
        try:
            result = DeepFace.represent(
                img_path      = img_path,
                model_name    = model_name,
                detector_backend = detector,
                enforce_detection = False,   # Don't crash on low-quality photos
                align         = True
            )
            # Take first detected face
            if result:
                embeddings[name] = np.array(result[0]["embedding"])
        except Exception as e:
            failed.append((name, str(e)))

    print(f"\n✅ Successfully embedded: {len(embeddings)} / {len(registry)} students")
    if failed:
        print(f"⚠️  Failed ({len(failed)}): {[n for n,_ in failed[:5]]}{'...' if len(failed)>5 else ''}")

    # Cache to disk
    with open(cache_path, 'wb') as f:
        pickle.dump(embeddings, f)
    print(f"💾 Embeddings cached to {cache_path}")
    return embeddings

face_embeddings = build_embedding_database(
    student_registry, MODEL_NAME, DETECTOR, EMBED_CACHE
)
print(f"\n📊 Database size: {len(face_embeddings)} face embeddings")


## 🔍 Step 7 — Face Recognition Engine

In [ ]:
def cosine_distance(a, b):
    """Cosine distance between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def recognize_face(img_path_or_array, embeddings_db, model_name,
                   detector, threshold, top_k=3):
    """
    Given an image, detect all faces and match each to the database.
    Returns a list of dicts: {bbox, name, distance, confidence, all_matches}
    """
    try:
        # Get all faces in image
        detected = DeepFace.represent(
            img_path_or_array,
            model_name       = model_name,
            detector_backend = detector,
            enforce_detection= False,
            align             = True
        )
    except Exception as e:
        print(f"Detection error: {e}")
        return []

    results = []
    for face_data in detected:
        query_emb = np.array(face_data["embedding"])
        region    = face_data.get("facial_area", {})

        # Compare to all students
        distances = {
            name: cosine_distance(query_emb, db_emb)
            for name, db_emb in embeddings_db.items()
        }
        sorted_matches = sorted(distances.items(), key=lambda x: x[1])
        best_name, best_dist = sorted_matches[0]

        if best_dist <= threshold:
            confidence = round((1 - best_dist) * 100, 1)
            identity   = best_name
        else:
            confidence = round((1 - best_dist) * 100, 1)
            identity   = "Unknown"

        results.append({
            "name"       : identity,
            "distance"   : round(best_dist, 4),
            "confidence" : confidence,
            "bbox"       : region,
            "top_matches": sorted_matches[:top_k]
        })

    return results

print("✅ Recognition engine ready")


## 📋 Step 8 — Attendance Logger

In [ ]:
# Initialize attendance CSV
def init_attendance_csv(csv_path, student_names):
    """Create a fresh attendance sheet with all students marked Absent."""
    today = datetime.now().strftime("%Y-%m-%d")
    df = pd.DataFrame({
        "Student Name": sorted(student_names),
        "Status"      : "Absent",
        "Time"        : "",
        "Date"        : today,
        "Confidence %": ""
    })
    df.to_csv(csv_path, index=False)
    return df

attendance_df = init_attendance_csv(ATTENDANCE_CSV, face_embeddings.keys())
print(f"✅ Attendance sheet initialized: {len(attendance_df)} students")
print(f"   Saved to: {ATTENDANCE_CSV}")

def mark_attendance(name, confidence, csv_path):
    """Mark a student as Present with current timestamp."""
    if name == "Unknown":
        return False
    df = pd.read_csv(csv_path)
    mask = df["Student Name"].str.lower() == name.lower()
    if mask.any():
        if df.loc[mask, "Status"].values[0] == "Present":
            return False  # Already marked
        now = datetime.now().strftime("%H:%M:%S")
        df.loc[mask, "Status"]       = "Present"
        df.loc[mask, "Time"]         = now
        df.loc[mask, "Confidence %"] = confidence
        df.to_csv(csv_path, index=False)
        return True
    return False

print("\n📝 Attendance logger functions ready")


## 🎨 Step 9 — Visualization Helper

In [ ]:
def visualize_recognition(img_path, results, title="Recognition Result"):
    """Draw bounding boxes and labels on the image."""
    img = cv2.imread(img_path) if isinstance(img_path, str) else img_path.copy()
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img_rgb)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')

    for r in results:
        bbox = r.get("bbox", {})
        x  = bbox.get("x", 0)
        y  = bbox.get("y", 0)
        w  = bbox.get("w", 0)
        h  = bbox.get("h", 0)
        name = r["name"]
        conf = r["confidence"]

        color = "#00FF41" if name != "Unknown" else "#FF3131"

        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=3, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)

        label = f"{name}\n{conf}%"
        ax.text(
            x, y - 10, label,
            color='white', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.85)
        )

    plt.tight_layout()
    plt.show()

print("✅ Visualizer ready")


## 🧪 Step 10 — Test Recognition on New Images
Upload one or more test images (photos of students). The system will:
- Detect all faces
- Match each face to the database
- Mark attendance automatically


In [ ]:
print("📤 Upload one or more test images to recognize...")
test_uploads = files.upload()

for test_filename, _ in test_uploads.items():
    print(f"\n{'='*60}")
    print(f"🔍 Processing: {test_filename}")

    results = recognize_face(
        test_filename, face_embeddings, MODEL_NAME, DETECTOR, THRESHOLD
    )

    if not results:
        print("⚠️  No faces detected in this image.")
        continue

    print(f"   Found {len(results)} face(s)")

    for i, r in enumerate(results):
        name = r["name"]
        dist = r["distance"]
        conf = r["confidence"]
        print(f"\n   Face {i+1}:")
        print(f"      Recognized as : {name}")
        print(f"      Confidence    : {conf}%")
        print(f"      Distance      : {dist}")

        # Top 3 matches
        print(f"      Top matches   :")
        for match_name, match_dist in r["top_matches"]:
            bar = "✅" if match_name == name and name != "Unknown" else "  "
            print(f"         {bar} {match_name:<30} dist={match_dist:.4f}")

        # Mark attendance
        if name != "Unknown":
            marked = mark_attendance(name, conf, ATTENDANCE_CSV)
            if marked:
                print(f"\n   ✅ Attendance MARKED for {name}")
            else:
                print(f"\n   ℹ️  {name} already marked or not in list")
        else:
            print(f"\n   ❓ Unknown face — not marked (confidence too low)")

    # Visualize
    visualize_recognition(test_filename, results, title=f"Recognition: {test_filename}")


## 📷 Step 11 — Live Webcam Auto-Attendance
The webcam streams continuously. When a face is detected and **stays visible for 30 seconds**, it is automatically captured and attendance is marked — no button needed.
- A live countdown timer shows how many seconds remain before capture
- The timer resets if the face disappears or changes
- Already-marked students are skipped automatically


In [ ]:
from IPython.display import Javascript, display as ipy_display
from google.colab.output import eval_js
from base64 import b64decode
import io, time, threading
from PIL import Image

# ── Configuration ────────────────────────────────────────────────────────────
DWELL_SECONDS   = 30      # seconds face must stay visible before auto-capture
SCAN_INTERVAL   = 3       # how often (seconds) Python re-checks the frame
STOP_AFTER_MARK = True    # stop webcam once attendance is marked

# ── Step 1: Inject the streaming webcam UI into the Colab output ────────────
webcam_js = Javascript('''
window._latestFrame   = null;   // latest JPEG data-URL
window._webcamRunning = true;

(async () => {
    // ── UI container ────────────────────────────────────────────────────────
    const wrap = document.createElement('div');
    wrap.id = 'wc-wrap';
    wrap.style.cssText = 'font-family:sans-serif;max-width:520px;margin:10px 0;';

    // Header
    const hdr = document.createElement('div');
    hdr.innerHTML = '<b>📷 Auto-Attendance Webcam</b>';
    hdr.style.cssText = 'background:#1a1a2e;color:#e0e0e0;padding:10px 14px;border-radius:8px 8px 0 0;font-size:15px;';
    wrap.appendChild(hdr);

    // Video element
    const video = document.createElement('video');
    video.setAttribute('autoplay','');
    video.setAttribute('playsinline','');
    video.style.cssText = 'width:100%;display:block;background:#000;';
    wrap.appendChild(video);

    // Status bar
    const status = document.createElement('div');
    status.id = 'wc-status';
    status.style.cssText = 'background:#16213e;color:#a0a0c0;padding:8px 14px;font-size:13px;min-height:28px;';
    status.textContent = '⏳ Initialising camera…';
    wrap.appendChild(status);

    // Countdown bar
    const barWrap = document.createElement('div');
    barWrap.style.cssText = 'background:#0f3460;height:6px;border-radius:0 0 8px 8px;overflow:hidden;';
    const barFill = document.createElement('div');
    barFill.id = 'wc-bar';
    barFill.style.cssText = 'height:6px;width:0%;background:#00b4d8;transition:width 1s linear;';
    barWrap.appendChild(barFill);
    wrap.appendChild(barWrap);

    document.body.appendChild(wrap);

    // ── Start camera ────────────────────────────────────────────────────────
    let stream;
    try {
        stream = await navigator.mediaDevices.getUserMedia({video:{width:640,height:480}});
    } catch(e) {
        status.textContent = '❌ Camera access denied: ' + e.message;
        return;
    }
    video.srcObject = stream;
    status.textContent = '👁️ Camera active — show your face and hold still for 30 s';

    // ── Canvas for frame capture ─────────────────────────────────────────────
    const canvas = document.createElement('canvas');

    // Capture a frame every second and store as data-URL
    const frameTick = setInterval(() => {
        if (!window._webcamRunning) { clearInterval(frameTick); return; }
        canvas.width  = video.videoWidth  || 640;
        canvas.height = video.videoHeight || 480;
        canvas.getContext('2d').drawImage(video, 0, 0);
        window._latestFrame = canvas.toDataURL('image/jpeg', 0.85);
    }, 1000);

    // ── Countdown display (driven by Python via setCountdown) ────────────────
    window.setCountdown = (secsLeft, total, msg) => {
        if (secsLeft <= 0) {
            status.textContent  = msg || '✅ Captured!';
            barFill.style.width = '100%';
            barFill.style.background = '#06d6a0';
        } else {
            const pct = ((total - secsLeft) / total * 100).toFixed(1);
            barFill.style.width = pct + '%';
            status.textContent  = msg || `⏱️ Hold still… ${secsLeft}s remaining`;
        }
    };

    window.stopWebcam = () => {
        window._webcamRunning = false;
        clearInterval(frameTick);
        stream.getTracks().forEach(t => t.stop());
        status.textContent = '✅ Webcam stopped. Attendance marked!';
        barFill.style.background = '#06d6a0';
        barFill.style.width = '100%';
    };
})();
''')  # end Javascript string

ipy_display(webcam_js)
time.sleep(3)   # let camera initialise

# ── Step 2: Python polling loop ──────────────────────────────────────────────
def get_latest_frame():
    """Pull the latest webcam frame from the browser."""
    data_url = eval_js('window._latestFrame')
    if not data_url:
        return None
    binary = b64decode(data_url.split(',')[1])
    img = Image.open(io.BytesIO(binary)).convert('RGB')
    path = '/content/webcam_latest.jpg'
    img.save(path)
    return path

def update_ui(secs_left, total, msg=None):
    eval_js(f'window.setCountdown({secs_left}, {total}, {repr(msg)})')

def quick_has_face(img_path):
    """Fast check: does this frame contain a face? Uses opencv haar for speed."""
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = cascade.detectMultiScale(img, scaleFactor=1.1, minNeighbors=5, minSize=(80,80))
    return len(faces) > 0

print('🎥 Webcam streaming. Auto-capture fires after face is visible for '
      f'{DWELL_SECONDS} seconds continuously…')
print('   (Python polls every', SCAN_INTERVAL, 'seconds)\n')

dwell_start   = None    # when the current face-dwell began
last_name     = None    # identity confirmed last scan
marked_names  = set()   # already-marked this session

try:
    while eval_js('window._webcamRunning'):
        time.sleep(SCAN_INTERVAL)

        frame_path = get_latest_frame()
        if frame_path is None:
            update_ui(DWELL_SECONDS, DWELL_SECONDS, '⏳ Waiting for frame…')
            continue

        # ── Fast face-presence check ─────────────────────────────────────────
        if not quick_has_face(frame_path):
            dwell_start = None
            last_name   = None
            update_ui(DWELL_SECONDS, DWELL_SECONDS, '👁️ No face detected — show your face')
            continue

        # ── Full recognition (every SCAN_INTERVAL seconds) ───────────────────
        results = recognize_face(
            frame_path, face_embeddings, MODEL_NAME, DETECTOR, THRESHOLD
        )
        if not results:
            dwell_start = None
            last_name   = None
            update_ui(DWELL_SECONDS, DWELL_SECONDS, '❓ Face found but not recognised')
            continue

        recognised = results[0]['name']
        conf       = results[0]['confidence']

        if recognised == 'Unknown':
            dwell_start = None
            last_name   = None
            update_ui(DWELL_SECONDS, DWELL_SECONDS,
                      f'❓ Unknown face ({conf}%) — not in database')
            continue

        if recognised in marked_names:
            update_ui(0, DWELL_SECONDS,
                      f'✅ {recognised} already marked present')
            if STOP_AFTER_MARK:
                break
            continue

        # ── Dwell timer ──────────────────────────────────────────────────────
        now = time.time()
        if recognised != last_name:          # face identity changed → reset
            dwell_start = now
            last_name   = recognised
            print(f'   👤 Detected: {recognised} ({conf}%) — starting 30-second timer')

        elapsed   = now - dwell_start
        remaining = max(0, DWELL_SECONDS - elapsed)
        update_ui(int(remaining), DWELL_SECONDS,
                  f'⏱️ {recognised} ({conf}%) — hold still: {int(remaining)}s left')

        if elapsed >= DWELL_SECONDS:
            # ── AUTO-CAPTURE ─────────────────────────────────────────────────
            capture_path = f'/content/capture_{recognised.replace(" ","_")}.jpg'
            import shutil as _sh
            _sh.copy(frame_path, capture_path)

            marked = mark_attendance(recognised, conf, ATTENDANCE_CSV)
            marked_names.add(recognised)

            print(f'\n   🟢 AUTO-CAPTURED: {recognised}')
            print(f'      Confidence : {conf}%')
            print(f'      Saved to   : {capture_path}')
            print(f'      Attendance : {"MARKED ✅" if marked else "already marked ℹ️"}')

            visualize_recognition(capture_path, results,
                                   title=f'Auto-Captured: {recognised}')

            dwell_start = None
            last_name   = None

            if STOP_AFTER_MARK:
                eval_js('window.stopWebcam()')
                break
            else:
                # Continue for next student
                time.sleep(3)

except KeyboardInterrupt:
    eval_js('window.stopWebcam()')
    print('\n⏹️ Webcam stopped manually.')

print('\n📋 Session summary:')
if marked_names:
    for n in marked_names:
        print(f'   ✅ {n}')
else:
    print('   No attendance marked this session.')


## 🔄 Step 12 — Batch Self-Test (Verify Database Accuracy)
Tests the model on its own registration photos to verify accuracy.
Useful for spotting students whose photo quality is too low.


In [ ]:
print("🔄 Running batch self-test on all student photos...")
print("   (Tests each student's photo against the full database)\n")

correct = 0
wrong   = 0
no_face = 0
results_log = []

for true_name, img_path in tqdm(student_registry.items(), desc="Testing"):
    ext = os.path.splitext(img_path)[1].lower()
    if ext == '.pdf':
        continue
    try:
        preds = recognize_face(
            img_path, face_embeddings, MODEL_NAME, DETECTOR, THRESHOLD
        )
        if not preds:
            no_face += 1
            results_log.append({"Student": true_name, "Predicted": "NO FACE", "Result": "❌"})
            continue

        pred_name = preds[0]["name"]
        conf      = preds[0]["confidence"]

        if pred_name.lower() == true_name.lower():
            correct += 1
            results_log.append({"Student": true_name, "Predicted": pred_name,
                                 "Confidence": conf, "Result": "✅"})
        else:
            wrong += 1
            results_log.append({"Student": true_name, "Predicted": pred_name,
                                 "Confidence": conf, "Result": "⚠️"})
    except Exception as e:
        no_face += 1
        results_log.append({"Student": true_name, "Predicted": f"ERROR: {e}", "Result": "❌"})

total    = correct + wrong + no_face
accuracy = round(correct / (total - no_face) * 100, 1) if (total - no_face) > 0 else 0

print(f"\n{'='*55}")
print(f"  SELF-TEST RESULTS")
print(f"  Total students  : {total}")
print(f"  ✅ Correct       : {correct}")
print(f"  ⚠️  Wrong         : {wrong}")
print(f"  ❌ No face found : {no_face}")
print(f"  📊 Accuracy      : {accuracy}%")
print(f"{'='*55}")

results_df = pd.DataFrame(results_log)
display(results_df[results_df["Result"] != "✅"].head(20))  # Show failures only


## 📊 Step 13 — View Attendance Report

In [ ]:
att_df = pd.read_csv(ATTENDANCE_CSV)

present = att_df[att_df["Status"] == "Present"]
absent  = att_df[att_df["Status"] == "Absent"]

print(f"📊 ATTENDANCE REPORT — {datetime.now().strftime('%Y-%m-%d')}")
print(f"{'='*55}")
print(f"  Total students : {len(att_df)}")
print(f"  ✅ Present      : {len(present)}")
print(f"  ❌ Absent       : {len(absent)}")
print(f"  📈 Attendance % : {round(len(present)/len(att_df)*100,1)}%")
print(f"{'='*55}\n")

# Visual summary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(
    [len(present), len(absent)],
    labels=["Present", "Absent"],
    colors=["#2ecc71", "#e74c3c"],
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 13}
)
axes[0].set_title("Attendance Distribution", fontsize=14, fontweight='bold')

# Bar chart – present students with confidence
if len(present) > 0:
    present_sorted = present.sort_values("Student Name")
    conf_vals = pd.to_numeric(present_sorted["Confidence %"], errors='coerce').fillna(0)
    axes[1].barh(present_sorted["Student Name"], conf_vals, color="#2ecc71", edgecolor='white')
    axes[1].set_xlabel("Confidence %")
    axes[1].set_title("Present Students & Confidence", fontsize=14, fontweight='bold')
    axes[1].set_xlim(0, 105)
    axes[1].tick_params(axis='y', labelsize=8)
else:
    axes[1].text(0.5, 0.5, "No students marked present yet",
                 ha='center', va='center', transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout()
plt.savefig("/content/attendance_chart.png", dpi=120, bbox_inches='tight')
plt.show()

print("\n✅ Present Students:")
display(present[["Student Name","Time","Confidence %"]].reset_index(drop=True))

print("\n❌ Absent Students:")
display(absent[["Student Name"]].reset_index(drop=True))


## 💾 Step 14 — Download Attendance Sheet

In [ ]:
# Save final attendance with timestamp in filename
final_name = f"attendance_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
shutil.copy(ATTENDANCE_CSV, f"/content/{final_name}")

print(f"📥 Downloading attendance CSV: {final_name}")
files.download(f"/content/{final_name}")

print(f"\n📥 Downloading attendance chart...")
files.download("/content/attendance_chart.png")

print("\n✅ Downloads complete!")


## ✏️ Step 15 — Manually Mark / Override Attendance (Optional)
Use this cell if you need to manually mark or correct any student.


In [ ]:
# ── Edit these variables as needed ──────────────────────────────────────────
MANUAL_NAME   = "Aniket Soni"      # Exact name (title case)
MANUAL_STATUS = "Present"          # "Present" or "Absent"
MANUAL_NOTE   = "Manual entry"     # Optional note

df = pd.read_csv(ATTENDANCE_CSV)
mask = df["Student Name"].str.lower() == MANUAL_NAME.lower()

if mask.any():
    df.loc[mask, "Status"]       = MANUAL_STATUS
    df.loc[mask, "Time"]         = datetime.now().strftime("%H:%M:%S") if MANUAL_STATUS == "Present" else ""
    df.loc[mask, "Confidence %"] = MANUAL_NOTE
    df.to_csv(ATTENDANCE_CSV, index=False)
    print(f"✅ Manually set {MANUAL_NAME} → {MANUAL_STATUS}")
else:
    print(f"⚠️  Student '{MANUAL_NAME}' not found in registry.")
    # Show similar names
    similar = [n for n in df["Student Name"] if MANUAL_NAME.split()[0].lower() in n.lower()]
    if similar:
        print(f"   Did you mean: {similar[:5]}")
